# Lab 20 — Drift detection and agent-as-judge calibration

Two halves. Half A simulates 30 days of eval scores with three drift patterns and detects each with the four canonical statistical tests. Half B runs an LLM-as-judge against a small human gold set, measures bias, applies mitigations, and computes Cohen's kappa.

Self-contained — all data is synthetic and deterministic. The patterns transfer directly to real production score streams from Lab 19's online evaluators.

> 📖 Required reading: [`concepts/evaluation/drift-detection.md`](../../concepts/evaluation/drift-detection.md), [`concepts/evaluation/agent-as-judge-calibration.md`](../../concepts/evaluation/agent-as-judge-calibration.md).
> ⬅️ Recommended: [Lab 19](../19-online-evaluation-and-sampling/) — produces the score streams this lab monitors.
> 🛠 All computation local; no API keys required.
> ⏱ Run time: 90-110 min including reading.


## Step 0: Setup

All dependencies already in the repo's pinned base. Deterministic seed for reproducibility.

In [ ]:
import numpy as np
from scipy.stats import beta, ks_2samp, wasserstein_distance, chisquare
from sklearn.metrics import cohen_kappa_score
import matplotlib.pyplot as plt

# Deterministic seed — same numbers across runs
RNG = np.random.default_rng(42)
print(f"numpy:  {np.__version__}")
print("matplotlib loaded")
print("  scipy.stats: ks_2samp, wasserstein_distance, chisquare")
print("  sklearn.metrics: cohen_kappa_score")


## ── Half A: Drift detection on eval scores ──

Module 4 produced a continuous stream of evaluator scores. Half A detects when that stream's distribution is changing — before the change becomes a user-facing regression.

### Step 1: Baseline score distribution

A well-functioning eval typically produces scores clustered toward the high end. We use `Beta(8, 2)` as the baseline: mean ≈ 0.8, mostly concentrated above 0.7. This represents "week 1 in production: the agent is performing well."

```
Beta(8, 2):  mean = 0.8, mode = 0.875, narrow distribution
```

In [ ]:
N_BASELINE = 1000

baseline = beta.rvs(a=8, b=2, size=N_BASELINE, random_state=RNG)

print("Baseline distribution: Beta(8, 2)")
print(f"  N samples: {N_BASELINE}")
print(f"  mean:      {baseline.mean():.3f}")
print(f"  std:       {baseline.std():.3f}")
print(f"  min/max:   {baseline.min():.3f} / {baseline.max():.3f}")


### Step 2: Three drift scenarios

Three patterns model the production reality:

- **Scenario A — gradual drift**: `Beta(6, 3)`, mean ≈ 0.67. Subtle but detectable. Models a slow input-distribution shift (your user base is changing).
- **Scenario B — abrupt drift**: `Beta(4, 6)`, mean ≈ 0.4. Large shift. Models the "model provider silently updated weights" scenario.
- **Scenario C — variance/shape drift**: `Beta(20, 5)`, mean ≈ 0.8 (same as baseline) but distribution narrows. Same mean; collapsed shape. This is the case naive mean-monitoring fails on.

In [ ]:
N_CURRENT = 1000

# Scenario A: gradual mean shift (subtle)
scenario_a = beta.rvs(a=6, b=3, size=N_CURRENT, random_state=RNG)

# Scenario B: abrupt mean shift (large)
scenario_b = beta.rvs(a=4, b=6, size=N_CURRENT, random_state=RNG)

# Scenario C: shape shift only — same mean, different variance
# Beta(20, 5) has mean = 20/25 = 0.8 (same as baseline!) but much narrower
scenario_c = beta.rvs(a=20, b=5, size=N_CURRENT, random_state=RNG)

print(f"{'Scenario':<30} {'Mean':>8} {'Std':>8} {'Note'}")
print("─" * 80)
print(f"{'Baseline (Beta 8,2)':<30} {baseline.mean():>8.3f} {baseline.std():>8.3f}  reference")
print(f"{'A - gradual drift':<30} {scenario_a.mean():>8.3f} {scenario_a.std():>8.3f}  mean shifted down")
print(f"{'B - abrupt drift':<30} {scenario_b.mean():>8.3f} {scenario_b.std():>8.3f}  large mean shift")
print(f"{'C - shape drift':<30} {scenario_c.mean():>8.3f} {scenario_c.std():>8.3f}  SAME mean, narrower")


In [ ]:
# Visualize the four distributions side by side
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharex=True, sharey=True)

for ax, data, title in zip(
    axes,
    [baseline, scenario_a, scenario_b, scenario_c],
    ["Baseline\nBeta(8,2)", "Scenario A\nGradual drift", "Scenario B\nAbrupt drift", "Scenario C\nShape drift"],
    strict=True,
):
    ax.hist(data, bins=30, density=True, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("score")
    ax.axvline(data.mean(), color="red", linestyle="--", lw=1, label=f"mean={data.mean():.2f}")
    ax.legend(fontsize=8)

axes[0].set_ylabel("density")
plt.tight_layout()
plt.show()


### Step 3: KS-test on each scenario

`scipy.stats.ks_2samp` returns a KS statistic D (the maximum vertical distance between empirical CDFs) and a p-value. Null hypothesis: the two samples come from the same distribution. We reject the null (declare drift) when p < 0.05.

The KS-test is nonparametric and catches *any* distributional shift — mean, variance, shape — not just mean changes.

In [ ]:
print(f"{'Scenario':<30} {'KS statistic':>14} {'p-value':>14} {'Reject H0 (p<0.05)?'}")
print("─" * 90)

for name, data in [
    ("A - gradual drift",   scenario_a),
    ("B - abrupt drift",    scenario_b),
    ("C - shape drift",     scenario_c),
]:
    result = ks_2samp(baseline, data)
    reject = "YES (drift)" if result.pvalue < 0.05 else "no"
    print(f"{name:<30} {result.statistic:>14.4f} {result.pvalue:>14.2e}   {reject}")


**KS catches all three.** Even Scenario C (same-mean, different-shape) produces a clear drift signal because KS measures CDF shape, not just mean. This is what naive mean-monitoring misses.

### Step 4: PSI from first principles

PSI (Population Stability Index) bins both distributions, then computes:

$$\text{PSI} = \sum_i (p_i^{\text{current}} - p_i^{\text{baseline}}) \cdot \ln\left(\frac{p_i^{\text{current}}}{p_i^{\text{baseline}}}\right)$$

Standard interpretation, used across financial ML for two decades:

- **PSI < 0.1**: stable, no action
- **0.1 ≤ PSI < 0.25**: moderate drift, investigate
- **PSI ≥ 0.25**: significant drift, alert

We compute it from scratch so the math is visible.

In [ ]:
def psi(baseline_samples, current_samples, n_bins=10):
    """Population Stability Index between two samples."""
    breakpoints = np.linspace(0, 1, n_bins + 1)

    baseline_counts, _ = np.histogram(baseline_samples, bins=breakpoints)
    current_counts, _  = np.histogram(current_samples,  bins=breakpoints)

    baseline_pct = baseline_counts / len(baseline_samples)
    current_pct  = current_counts  / len(current_samples)

    # Avoid log(0) — replace zero bins with small epsilon
    epsilon = 1e-6
    baseline_pct = np.where(baseline_pct == 0, epsilon, baseline_pct)
    current_pct  = np.where(current_pct  == 0, epsilon, current_pct)

    return float(np.sum((current_pct - baseline_pct) * np.log(current_pct / baseline_pct)))


def interpret_psi(value):
    if value < 0.1:
        return "stable"
    if value < 0.25:
        return "moderate drift — investigate"
    return "SIGNIFICANT DRIFT — alert"


print(f"{'Scenario':<30} {'PSI':>8}  {'Interpretation'}")
print("─" * 75)
for name, data in [
    ("A - gradual drift", scenario_a),
    ("B - abrupt drift",  scenario_b),
    ("C - shape drift",   scenario_c),
]:
    value = psi(baseline, data)
    print(f"{name:<30} {value:>8.3f}  {interpret_psi(value)}")


### Step 5: Wasserstein distance

Wasserstein (Earth Mover's Distance) measures the minimum amount of "work" to transform one distribution into the other. Unlike PSI, it's **scale-aware** — the value's magnitude is in the units of the metric.

For an eval score in [0, 1], a Wasserstein distance of 0.1 means roughly "the mean has shifted by about 0.1 in absolute terms." Easier to communicate to non-technical stakeholders than p-values.

In [ ]:
print(f"{'Scenario':<30} {'Wasserstein':>12} {'Approx mean shift'}")
print("─" * 70)
for name, data in [
    ("A - gradual drift", scenario_a),
    ("B - abrupt drift",  scenario_b),
    ("C - shape drift",   scenario_c),
]:
    distance = wasserstein_distance(baseline, data)
    print(f"{name:<30} {distance:>12.4f}  {abs(baseline.mean() - data.mean()):.3f}")


Notice Scenario C: Wasserstein is small (means are nearly equal) but KS p-value was tiny. Wasserstein and KS are answering different questions — Wasserstein quantifies the magnitude of the mean shift in score units; KS asks "are the distributions the same shape at all."

For drift detection on shape changes (like Scenario C), KS is the right test. For drift detection where you want a magnitude in score units (for alerting thresholds), Wasserstein is the right test. Use both.

### Step 6: Which test catches which scenario

Summary table for the decision boundary:

In [ ]:
def summary_row(name, data):
    ks_result = ks_2samp(baseline, data)
    return {
        "scenario": name,
        "baseline_mean": baseline.mean(),
        "current_mean":  data.mean(),
        "KS_p":          ks_result.pvalue,
        "KS_reject":     "YES" if ks_result.pvalue < 0.05 else "no",
        "PSI":           psi(baseline, data),
        "Wasserstein":   wasserstein_distance(baseline, data),
    }


rows = [
    summary_row("A - gradual",  scenario_a),
    summary_row("B - abrupt",   scenario_b),
    summary_row("C - shape",    scenario_c),
]

print(f"{'Scenario':<15} {'mean Δ':>8} {'KS p':>10} {'KS?':>5} {'PSI':>7} {'Wasser':>8}")
print("─" * 65)
for r in rows:
    mean_delta = r["current_mean"] - r["baseline_mean"]
    print(f"{r['scenario']:<15} {mean_delta:>+8.3f} {r['KS_p']:>10.2e} {r['KS_reject']:>5} {r['PSI']:>7.3f} {r['Wasserstein']:>8.4f}")

print()
print("Takeaways:")
print("- KS catches all three scenarios (it's distribution-shape-aware).")
print("- Wasserstein agrees with mean shift magnitude but UNDERESTIMATES Scenario C")
print("  (which has same mean, different shape).")
print("- PSI catches A and B clearly; for Scenario C, PSI is also large because binning")
print("  exposes the shape difference.")
print("- For shape-only drift like Scenario C: use KS. Mean-monitoring would miss it.")


### Step 7: Rolling-window drift detection

Production needs a stream, not a snapshot. The rolling-window pattern: fix a baseline, slide a window of recent samples, compute KS at every interval, plot the p-value over time. When p-value drops below threshold for N consecutive windows, alert.

We simulate a 1000-sample score stream with a drift event injected at sample 500.

In [ ]:
# Construct a 1000-sample stream with a mid-stream drift event
stream_pre  = beta.rvs(a=8, b=2, size=500, random_state=RNG)    # pre-drift
stream_post = beta.rvs(a=4, b=6, size=500, random_state=RNG)    # post-drift (abrupt)
stream = np.concatenate([stream_pre, stream_post])

# Rolling-window KS-test: slide a 100-sample window across the stream
WINDOW_SIZE = 100
STEP = 25

window_centers = []
p_values = []
ks_statistics = []

for start in range(0, len(stream) - WINDOW_SIZE + 1, STEP):
    window = stream[start : start + WINDOW_SIZE]
    result = ks_2samp(baseline, window)
    window_centers.append(start + WINDOW_SIZE // 2)
    p_values.append(result.pvalue)
    ks_statistics.append(result.statistic)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

ax1.plot(window_centers, p_values, marker="o", markersize=3, linewidth=1)
ax1.axhline(0.05, color="orange", linestyle="--", label="warning (p=0.05)")
ax1.axhline(0.001, color="red",    linestyle="--", label="critical (p=0.001)")
ax1.axvline(500, color="black", linestyle=":", alpha=0.5, label="actual drift event")
ax1.set_yscale("log")
ax1.set_ylabel("KS p-value (log)")
ax1.set_title("Rolling-window KS p-value across the score stream")
ax1.legend(loc="lower left", fontsize=8)
ax1.grid(alpha=0.3)

ax2.plot(window_centers, ks_statistics, marker="o", markersize=3, color="darkblue", linewidth=1)
ax2.axvline(500, color="black", linestyle=":", alpha=0.5)
ax2.set_xlabel("sample position in stream")
ax2.set_ylabel("KS statistic D")
ax2.set_title("KS statistic (effect size) — independent of sample size")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nDetection latency: the drift event was at sample 500.")
print(f"The p-value crossed the critical threshold (p<0.001) at approximately sample {500 + WINDOW_SIZE // 2}.")
print("This is roughly one window-width after the actual drift — the cost of needing a window of post-drift data.")


## ── Half B: LLM-as-judge calibration via Cohen's kappa ──

Half A monitors score distributions for drift. But the scores themselves need to actually mean what their label says they mean. Half B addresses that: calibrate the judge against human ground truth via Cohen's kappa.

### Step 8: The human gold set

A small human-labeled gold set is the anchor that lets you measure judge quality. The lab uses 10 examples for visibility; production gold sets are typically 50-200.

Each example has an agent answer + a human binary label for whether the answer is well-grounded (1) or not (0).

In [ ]:
# Human-labeled gold set
# Each example: (answer_text, human_label_0_or_1)
# Mix of correct/incorrect, short/long — to expose verbosity bias later
gold_set = [
    # Short correct (well-grounded)
    ("MCP is an open standard [1].",                                          1),
    ("LangGraph is a stateful runtime [1].",                                  1),

    # Long correct (well-grounded but verbose)
    ("MCP is an open standard [1] that has seen broad production adoption across many platforms [2], with extensive documentation [3] and a growing developer community [4].", 1),
    ("Modern observability stacks like LangSmith [1], Phoenix [2], and Langfuse [3] all support OpenTelemetry-based tracing for agent workflows.", 1),

    # Short incorrect (no citations)
    ("MCP is some kind of protocol thing.",                                   0),
    ("LangGraph does agent stuff.",                                            0),

    # Long incorrect (verbose with fabricated citations)
    ("MCP is described extensively in academic literature [47] and has been the subject of numerous research papers [88] published in top venues [99] over the past several years [73].", 0),
    ("The system architecture relies on advanced concepts [42] that were originally proposed in seminal works [56] and refined through iterative experimentation [91].", 0),

    # Edge cases
    ("MCP supports tool integration.",                                         0),  # short, plausible but ungrounded
    ("LangSmith [1] integrates with LangChain [2] for tracing.",               1),  # short, correctly grounded
]

human_labels = [label for (_, label) in gold_set]
print(f"Gold set: {len(gold_set)} examples")
print(f"  Positive (well-grounded): {sum(human_labels)}")
print(f"  Negative (not grounded):  {len(human_labels) - sum(human_labels)}")


### Step 9: A simulated LLM judge with controlled biases

In a real lab you would call OpenAI/Anthropic to score each example. To keep this self-contained and deterministic, we simulate a judge as a Python function with realistic biases:

- **Verbosity bias**: longer answers score higher, independent of correctness.
- **Marker-count signal**: the judge correctly notices citation markers, but weighs length more than it should.

This is a faithful simulation of how a poorly-calibrated production judge behaves. Real judges have these biases unless explicitly mitigated.

In [ ]:
import re


def llm_judge_naive(answer: str) -> int:
    """A biased LLM-as-judge simulation.

    Real judges weigh length more than they should (verbosity bias).
    This function captures that: longer answers get a boost.
    Returns 0 or 1 (binary judgment of well-grounded-ness).
    """
    markers = len(re.findall(r"\[\d+\]", answer))
    length = len(answer.split())

    # Weighted score: markers matter but length matters more than it should
    score = markers * 0.3 + length * 0.02

    # Threshold to binary
    return 1 if score > 1.0 else 0


# Score the gold set
judge_labels = [llm_judge_naive(answer) for (answer, _) in gold_set]

print(f"{'Answer (first 60 chars)':<63} {'Human':>6} {'Judge':>6}")
print("─" * 80)
for (answer, human), judge in zip(gold_set, judge_labels, strict=True):
    short = answer[:60] + ("..." if len(answer) > 60 else "")
    print(f"{short:<63} {human:>6} {judge:>6}")


### Step 10: Cohen's kappa baseline

Cohen's kappa measures inter-rater agreement *corrected for chance*. Two raters flipping coins independently agree 50% of the time on binary labels by random chance alone — kappa subtracts that off.

The Landis & Koch 1977 interpretation ranges (still the field's convention):

| Kappa | Interpretation |
|---|---|
| < 0 | Worse than chance |
| 0.00 – 0.20 | Slight |
| 0.20 – 0.40 | Fair |
| 0.40 – 0.60 | Moderate |
| 0.60 – 0.80 | Substantial |
| 0.80 – 1.00 | Almost perfect |

**Substantial agreement (κ ≥ 0.6) is the common production target.**

In [ ]:
def interpret_kappa(k):
    if k < 0:
        return "worse than chance — something is broken"
    if k < 0.2:
        return "slight (don't trust)"
    if k < 0.4:
        return "fair (aggregate trends only)"
    if k < 0.6:
        return "moderate (usable, review edge cases)"
    if k < 0.8:
        return "substantial (production target)"
    return "almost perfect"


kappa_baseline = cohen_kappa_score(human_labels, judge_labels)
print(f"Baseline Cohen's kappa: {kappa_baseline:.3f}")
print(f"  Interpretation: {interpret_kappa(kappa_baseline)}")
print()
print(f"Raw accuracy: {sum(h == j for h, j in zip(human_labels, judge_labels, strict=True)) / len(judge_labels):.2f}")
print("  Note: kappa is more informative than accuracy because it corrects for chance.")


### Step 11: Measure-and-mitigate verbosity bias

The naive judge correlates score with length. We can measure this and then apply a length-controlled rubric that explicitly tells the judge not to use length as a signal.

**Measurement**: pair short-correct/long-correct examples; see if the judge over-weights length.

**Mitigation**: a length-controlled judge that uses *only* marker count, not length.

In [ ]:
# Examine the bias quantitatively
def analyze_length_bias(gold_set, judge_fn):
    """Compute mean judge score split by answer length quartile."""
    lengths = [len(a.split()) for (a, _) in gold_set]
    median_length = np.median(lengths)

    short_correct = [judge_fn(a) for (a, h) in gold_set if len(a.split()) <= median_length and h == 1]
    long_correct  = [judge_fn(a) for (a, h) in gold_set if len(a.split()) >  median_length and h == 1]
    short_wrong   = [judge_fn(a) for (a, h) in gold_set if len(a.split()) <= median_length and h == 0]
    long_wrong    = [judge_fn(a) for (a, h) in gold_set if len(a.split()) >  median_length and h == 0]

    return {
        "short_correct_mean": np.mean(short_correct) if short_correct else None,
        "long_correct_mean":  np.mean(long_correct)  if long_correct  else None,
        "short_wrong_mean":   np.mean(short_wrong)   if short_wrong   else None,
        "long_wrong_mean":    np.mean(long_wrong)    if long_wrong    else None,
    }


naive_analysis = analyze_length_bias(gold_set, llm_judge_naive)
print("Naive judge — score by length × correctness:")
print(f"  Short correct: {naive_analysis['short_correct_mean']:.2f}  (target: 1.0)")
print(f"  Long correct:  {naive_analysis['long_correct_mean']:.2f}  (target: 1.0)")
print(f"  Short wrong:   {naive_analysis['short_wrong_mean']:.2f}  (target: 0.0)")
print(f"  Long wrong:    {naive_analysis['long_wrong_mean']:.2f}  (target: 0.0)")
print()
print("→ The 'Long wrong' score is inflated by length; that's verbosity bias.")
print("→ Short-correct often scores 0 because length is too small to clear the threshold.")


In [ ]:
def llm_judge_length_controlled(answer: str) -> int:
    """Length-controlled judge.

    Mitigation: explicit rubric ignores length; relies only on marker count
    and a sanity check that markers don't look fabricated (number > 10).
    """
    markers = re.findall(r"\[(\d+)\]", answer)
    n_markers = len(markers)
    max_marker = max((int(m) for m in markers), default=0)

    if n_markers == 0:
        return 0
    if max_marker > 10:
        return 0  # fabrication signal — high marker numbers
    return 1


judge_labels_mitigated = [llm_judge_length_controlled(answer) for (answer, _) in gold_set]
kappa_mitigated = cohen_kappa_score(human_labels, judge_labels_mitigated)

print("After verbosity-bias mitigation:")
print(f"  Cohen's kappa: {kappa_mitigated:.3f}  ({interpret_kappa(kappa_mitigated)})")
print(f"  Improvement:   +{kappa_mitigated - kappa_baseline:.3f}")
print()
print(f"{'Answer (first 60 chars)':<63} {'Human':>6} {'Naive':>6} {'Fixed':>6}")
print("─" * 90)
for (answer, human), naive, fixed in zip(gold_set, judge_labels, judge_labels_mitigated, strict=True):
    short = answer[:60] + ("..." if len(answer) > 60 else "")
    marker = " " if fixed == human else "✗"
    print(f"{short:<63} {human:>6} {naive:>6} {fixed:>6} {marker}")


### Step 12: The recalibration loop over simulated time

A single kappa measurement isn't very informative. The trend is the signal.

We simulate 12 weeks of judge runs. At week 6, we inject a synthetic judge-drift event (the judge starts being more lenient — possibly due to a silent model update). The calibration loop catches this when kappa crosses below the 0.6 substantial-agreement threshold.

In [ ]:
N_WEEKS = 12
JUDGE_DRIFT_WEEK = 6

# Per-week kappa simulation
weekly_kappas = []
for week in range(1, N_WEEKS + 1):
    week_rng = np.random.default_rng(42 + week)

    if week < JUDGE_DRIFT_WEEK:
        # Pre-drift: judge is well-calibrated. Use the mitigated judge.
        labels = [llm_judge_length_controlled(answer) for (answer, _) in gold_set]
        # Add small noise — 10% chance of a single flip per week to simulate real variance
        if week_rng.random() < 0.3:
            flip_idx = week_rng.integers(0, len(labels))
            labels[flip_idx] = 1 - labels[flip_idx]
    else:
        # Post-drift: judge becomes more lenient — calls everything 1
        labels = [llm_judge_length_controlled(answer) for (answer, _) in gold_set]
        # Flip multiple negative labels to positive (judge over-rates after drift)
        for idx, (_, h) in enumerate(gold_set):
            if h == 0 and week_rng.random() < 0.5:
                labels[idx] = 1

    weekly_kappa = cohen_kappa_score(human_labels, labels)
    weekly_kappas.append(weekly_kappa)

# Plot
fig, ax = plt.subplots(figsize=(10, 4.5))
weeks = list(range(1, N_WEEKS + 1))
ax.plot(weeks, weekly_kappas, marker="o", markersize=8, linewidth=2, color="steelblue")
ax.axhline(0.6, color="orange", linestyle="--", label="substantial threshold (κ=0.6)")
ax.axhline(0.4, color="red",    linestyle="--", label="moderate threshold (κ=0.4)")
ax.axvline(JUDGE_DRIFT_WEEK - 0.5, color="black", linestyle=":", alpha=0.6, label=f"judge drift event (week {JUDGE_DRIFT_WEEK})")
ax.set_xlabel("week")
ax.set_ylabel("Cohen's kappa")
ax.set_title("Recalibration loop — kappa over simulated time")
ax.set_xticks(weeks)
ax.legend(loc="lower left", fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(-0.1, 1.05)
plt.tight_layout()
plt.show()

print("Week-by-week kappa:")
for w, k in zip(weeks, weekly_kappas, strict=True):
    marker = " ← below 0.6 substantial threshold" if k < 0.6 else ""
    print(f"  Week {w:2d}: κ = {k:.3f}{marker}")

print()
first_below = next((w for w, k in zip(weeks, weekly_kappas, strict=True) if k < 0.6), None)
if first_below:
    detection_lag = first_below - JUDGE_DRIFT_WEEK
    print(f"Drift event at week {JUDGE_DRIFT_WEEK}; first kappa below substantial threshold at week {first_below}.")
    print(f"Detection lag: {detection_lag} week(s).")


## Step 13: Synthesis — the Path 06 trust stack

What this lab built:

**Half A — Drift detection**:
- Generated baseline + three drift scenarios (gradual, abrupt, shape-only).
- KS-test, PSI (from first principles), Wasserstein distance — applied to each scenario.
- Showed that shape-only drift (Scenario C, same mean as baseline) defeats naive mean-monitoring; KS catches it.
- Rolling-window detector on a 1000-sample stream with mid-stream drift; KS p-value over time, with warning/critical thresholds and visible detection lag.

**Half B — Calibration**:
- Small human gold set (10 examples) with varied length and quality.
- Simulated naive LLM judge with verbosity bias; baseline Cohen's kappa.
- Length-controlled mitigation; improved kappa.
- 12-week recalibration loop with injected judge-drift event; kappa-over-time plot showing detection.

**The Path 06 trust stack assembled**:

| Layer | Module | What it gives you | What fails silently if you skip it |
|---|---|---|---|
| 1. Instrumentation | M2-3 | Traces with `gen_ai.*` attributes | No data to monitor; you're blind |
| 2. Online evaluators | M4 | Scores attached to traces | Aggregate quality is unknown |
| 3. Drift detection | M5 (this) | Alerts when scores shift | Silent degradations go unnoticed |
| 4. Calibration | M5 (this) | Trust that scores mean what they say | Drift alerts that don't correspond to real quality changes |

Each layer fails silently if a lower layer is broken. Drift detection on uncalibrated scores tells you something is shifting; it doesn't tell you whether the shift reflects real quality changes or judge drift. The calibration loop is what disambiguates.

**The complete production pattern**:

1. Tail-sample at the Collector (Module 4) → reduces ingestion cost.
2. Online evaluators score sampled traces (Module 4) → produces the score stream.
3. Drift detection runs on the score stream (Module 5, Half A) → fires alerts on distribution shifts.
4. Calibration loop validates judge quality weekly (Module 5, Half B) → distinguishes real drift from judge drift.

A skipped step compromises the trust of every step downstream.

What this lab didn't cover (deferred):

- **Embedding-space drift on RAG inputs.** Path 02 v2 territory.
- **Autoencoder reconstruction-loss drift.** Higher-complexity research-frontier; Arthur AI's approach.
- **Cost attribution on calibration runs.** Module 6.
- **Multi-turn (threaded) judge calibration.** Module 7.

Path 06 Module 5 is now complete. Module 6 (cost attribution + sampling at scale) and Module 7 (multi-turn evaluation) close Path 06 v1 in future batches.

✓ **Module 5 complete.**
